# LLM ile Metin Etiketleme (Annotation)
### Hesaplamalı Sosyal Bilimler — Uygulama Dersi

Bu notebook'ta şunları yapacağız:

1. **API'ye bağlanmak** (güvenli şekilde)
2. **Veri setini yüklemek** (200 sentetik tweet)
3. **Gradio arayüzü** üzerinden interaktif prompt engineering
4. **Tüm veri setini** sistematik olarak etiketlemek
5. **Sonuçları analiz etmek**

---
> **Veri seti hakkında:** Bu derste kullandığımız tweetler **sentetik** (yapay) verilerdir.  
> Gerçek kullanıcılara ait değil, araştırma amacıyla oluşturulmuştur.  
> Veri seti **push** (itici) ve **pull** (çekici) olmak üzere iki ana kategori,  
> ve 15 alt kategori içermektedir.

## 0. Kurulum

Gerekli kütüphaneleri yükleyelim. Terminal'de bir kez çalıştırmanız yeterli.

In [1]:
!pip install openai gradio pandas openpyxl -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Kütüphaneleri İçe Aktar

In [3]:
import os
import json
import time
import random
import getpass
import pandas as pd
import gradio as gr
from openai import OpenAI

print("✓ Kütüphaneler yüklendi.")

✓ Kütüphaneler yüklendi.


## 2. API Bağlantısı

API key'i **asla** koda düz metin olarak yazmayın.  
`getpass` ile girilen key ekranda görünmez.

In [5]:
# Güvenli key girişi — ekranda görünmez
api_key = getpass.getpass("OpenAI API key'inizi girin: ")

client = OpenAI(api_key=api_key)

# Bağlantıyı test et
test = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Merhaba"}],
    max_tokens=5
)
print("✓ API bağlantısı başarılı.")

OpenAI API key'inizi girin:  ········


✓ API bağlantısı başarılı.


## 3. Veri Setini Yükle

200 sentetik tweet içeren Excel dosyasını yüklüyoruz.  
Veri setinde `id`, `metin`, `ana_kategori` ve `alt_kategori` sütunları var.

In [7]:
import pandas as pd

# Tam dosya yolu
DOSYA_ADI = "/Users/helinekmen/Desktop/vao-yz2/VAO-case.xlsx"

# Excel dosyasını yükle
df = pd.read_excel(DOSYA_ADI)

print(f"✓ Veri seti yüklendi: {len(df)} tweet")
print(f"  Sütunlar: {df.columns.tolist()}")
print()

print("Ana kategori dağılımı:")
print(df["ana_kategori"].value_counts().to_string())
print()

print("Alt kategori dağılımı:")
print(df["alt_kategori"].value_counts().to_string())

✓ Veri seti yüklendi: 200 tweet
  Sütunlar: ['id', 'metin', 'ana_kategori', 'alt_kategori']

Ana kategori dağılımı:
ana_kategori
push    119
pull     81

Alt kategori dağılımı:
alt_kategori
kariyer_gelisimi           15
ekonomik_istikrarsizlik    14
liyakat_sorunu             14
ozgurluk_kisitlari         14
is_firsatlari              14
yuksek_yasam_kalitesi      14
gelecek_belirsizligi       13
girisimcilik_engelleri     13
egitim_sorunlari           13
hukuk_guvenlik_kaygisi     13
zayif_sosyal_guvence       13
daha_yuksek_gelir          13
cocuklar_icin_gelecek      13
dusuk_yasam_kalitesi       12
refah_devleti              12


In [9]:
# İlk 5 tweeti önizleyelim
df.head()

,id,metin,ana_kategori,alt_kategori
0,1,Maaşım enflasyona yetişemiyor. Geçen yıl aldığ...,push,ekonomik_istikrarsizlik
1,2,"Kira %200 arttı, maaş %30. Matematik yapmıyoru...",push,ekonomik_istikrarsizlik
2,3,Tasarruf etmeye çalışıyorum ama TL her gün eri...,push,ekonomik_istikrarsizlik
3,4,Aylık harcamalarımı excele döktüm. Artı kalem ...,push,ekonomik_istikrarsizlik
4,5,Dolar kuru her sabah farklı. Hayat planlamak i...,push,ekonomik_istikrarsizlik


## 4. Annotation Fonksiyonu

Bu fonksiyon tek bir tweeti verilen system prompt ve ayarlarla etiketler.  
Gradio arayüzü ve toplu etiketleme bu fonksiyonu kullanacak.

In [11]:
def tweet_etiketle(
    metin: str,
    system_prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0
) -> dict:
    """
    Tek bir tweeti etiketler ve sonucu sözlük olarak döndürür.

    Parametreler:
        metin         : Etiketlenecek tweet metni
        system_prompt : Modelin rol ve kural tanımı
        model         : Kullanılacak model adı
        temperature   : Çıktı çeşitliliği (0=tutarlı, 1=yaratıcı)

    Döndürür:
        Başarıda  → {'ana_kategori': ..., 'alt_kategori': ..., 'güven': ..., 'gerekçe': ...}
        Hata      → {'ana_kategori': 'HATA', 'alt_kategori': 'HATA', ...}
    """
    try:
        yanit = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": f"Tweet: {metin}"}
            ]
        )
        ham_cikti = yanit.choices[0].message.content.strip()

        # Modelin ürettiği JSON'u ayıkla (bazen ```json blokları içinde gelir)
        if "```" in ham_cikti:
            ham_cikti = ham_cikti.split("```")[1]
            if ham_cikti.startswith("json"):
                ham_cikti = ham_cikti[4:]

        sonuc = json.loads(ham_cikti)
        sonuc["ham_cikti"] = yanit.choices[0].message.content
        sonuc["token_kullanim"] = yanit.usage.total_tokens
        return sonuc

    except json.JSONDecodeError:
        return {
            "ana_kategori": "HATA", "alt_kategori": "HATA",
            "güven": None, "gerekçe": "Geçersiz JSON çıktısı",
            "ham_cikti": yanit.choices[0].message.content if 'yanit' in dir() else "",
            "token_kullanim": 0
        }
    except Exception as e:
        return {
            "ana_kategori": "HATA", "alt_kategori": "HATA",
            "güven": None, "gerekçe": str(e),
            "ham_cikti": "", "token_kullanim": 0
        }

print("✓ Annotation fonksiyonu tanımlandı.")

✓ Annotation fonksiyonu tanımlandı.


## 5. 🎛️ İnteraktif Prompt Engineering Arayüzü

Aşağıdaki Gradio arayüzü ile:
- Rastgele tweet çekip deneyin
- System prompt'unuzu düzenleyin
- Model ve temperature seçin
- Sonucu anında görün

**Amacınız:** Mevcut `ana_kategori` ve `alt_kategori` etiketleriyle modelin etiketlerini mümkün olduğunca örtüştürmek.

In [13]:
# Varsayılan system prompt — derste öğrenciler bunu düzenleyecek
VARSAYILAN_PROMPT = """Sen bir sosyal bilim araştırma asistanısın. Görevin, Türkiye'deki genç
ve nitelikli iş gücüne ait tweetleri göç motivasyonu kategorilerine göre etiketlemektir.

ANA KATEGORİLER:
- "push" : Türkiye'yi terk etmeye iten nedenler (olumsuz koşullar)
- "pull" : Yurt dışını çekici kılan nedenler (olumlu fırsatlar)

ALT KATEGORİLER (push):
- ekonomik_istikrarsizlik  : Enflasyon, kur, geçim sıkıntısı
- liyakat_sorunu           : Torpil, adam kayırma, fırsat eşitsizliği
- ozgurluk_kisitlari       : Siyasi baskı, ifade özgürlüğü, akademik özgürlük
- hukuk_guvenlik_kaygisi   : Hukuk devleti, yargı bağımsızlığı
- egitim_sorunlari         : Eğitim kalitesi, akademik ortam
- girisimcilik_engelleri   : Bürokratik engeller, iş kurma güçlüğü
- zayif_sosyal_guvence     : Sağlık, emeklilik, sosyal haklar
- gelecek_belirsizligi     : Uzun vadeli belirsizlik, planlayamama
- dusuk_yasam_kalitesi     : Kentsel yaşam, çevre, altyapı

ALT KATEGORİLER (pull):
- daha_yuksek_gelir        : Maaş farkı, döviz kazancı
- kariyer_gelisimi         : Kariyer fırsatları, mentorluk, büyüme
- is_firsatlari            : Pozisyon çeşitliliği, istihdam
- yuksek_yasam_kalitesi    : Altyapı, çevre, şehir kalitesi
- refah_devleti            : Sosyal güvenlik, sağlık, haklar
- cocuklar_icin_gelecek    : Çocukların eğitimi, geleceği

Cevabını SADECE şu JSON formatında ver, başka hiçbir şey yazma:
{"ana_kategori": "push veya pull", "alt_kategori": "yukarıdan birini seç", "güven": "yüksek/orta/düşük", "gerekçe": "max 2 cümle"}"""


# ─── Gradio arayüzü ───────────────────────────────────────────────────────────

def rastgele_tweet_getir():
    """Veri setinden rastgele bir tweet seçer ve gösterir."""
    satir = df.sample(1).iloc[0]
    return (
        str(satir["id"]),
        satir["metin"],
        f"Ana: {satir['ana_kategori']} | Alt: {satir['alt_kategori']}"
    )


def etiketle_ve_goster(tweet_metin, system_prompt, model_sec, temperature):
    """Seçili tweeti verilen ayarlarla etiketler ve sonucu formatlar."""
    if not tweet_metin.strip():
        return "⚠️ Önce 'Rastgele Tweet Getir' butonuna basın."

    sonuc = tweet_etiketle(
        metin=tweet_metin,
        system_prompt=system_prompt,
        model=model_sec,
        temperature=temperature
    )

    if sonuc["ana_kategori"] == "HATA":
        return f"❌ Hata: {sonuc['gerekçe']}"

    cikti = f"""### Sonuç

| Alan | Değer |
|------|-------|
| **Ana Kategori** | {sonuc.get('ana_kategori', '-')} |
| **Alt Kategori** | {sonuc.get('alt_kategori', '-')} |
| **Güven** | {sonuc.get('güven', '-')} |
| **Kullanılan Token** | {sonuc.get('token_kullanim', '-')} |

**Gerekçe:**  
{sonuc.get('gerekçe', '-')}

---
**Ham JSON çıktı (modelden gelen):**
```json
{sonuc.get('ham_cikti', '-')}
```"""
    return cikti


# ─── Arayüz tasarımı ─────────────────────────────────────────────────────────

with gr.Blocks(
    title="LLM Annotation Playground",
    theme=gr.themes.Soft(),
    css="""
        .header { text-align: center; margin-bottom: 10px; }
        .label-col { background: #f0f4ff; border-radius: 8px; padding: 10px; }
    """
) as arayuz:

    gr.Markdown(
        """# 🔬 LLM Annotation Playground
        **Amaç:** System prompt'u düzenleyerek modelin tahminlerini gerçek etiketlerle örtüştürün.
        """,
        elem_classes="header"
    )

    with gr.Row():

        # ── SOL KOLON: Girdi ve ayarlar ───────────────────────────────────────
        with gr.Column(scale=1):

            gr.Markdown("### ⚙️ Model Ayarları")

            model_sec = gr.Dropdown(
                label="Model Seçimi",
                choices=["gpt-4o-mini", "gpt-4o", "gpt-3.5-turbo"],
                value="gpt-4o-mini",
                info="Küçük model → hızlı ve ucuz. Büyük model → daha doğru ama maliyetli."
            )

            temperature = gr.Slider(
                label="Temperature",
                minimum=0.0,
                maximum=1.0,
                value=0.0,
                step=0.1,
                info="0 = tutarlı ve tahmin edilebilir | 1 = yaratıcı ve değişken"
            )

            gr.Markdown("### 📝 System Prompt")
            gr.Markdown(
                "Modele kim olduğunu ve nasıl davranacağını söylediğiniz yer. "
                "Deneyin: kategorileri kaldırın, ekleyin, tanımları değiştirin."
            )

            system_prompt_box = gr.Textbox(
                label="System Prompt (Düzenlenebilir)",
                value=VARSAYILAN_PROMPT,
                lines=20,
                max_lines=30,
                show_copy_button=True
            )

            varsayilana_don = gr.Button("↩ Varsayılan Prompt'a Dön", variant="secondary", size="sm")

        # ── SAĞ KOLON: Tweet ve sonuç ─────────────────────────────────────────
        with gr.Column(scale=1):

            gr.Markdown("### 🐦 Tweet")

            tweet_id_box = gr.Textbox(
                label="Tweet ID",
                interactive=False,
                max_lines=1
            )

            tweet_metin_box = gr.Textbox(
                label="Tweet Metni",
                lines=4,
                interactive=False,
                placeholder="Rastgele tweet getirmek için butona basın..."
            )

            gercek_etiket_box = gr.Textbox(
                label="✅ Gerçek Etiket (Veri Setindeki)",
                interactive=False,
                elem_classes="label-col"
            )

            with gr.Row():
                rastgele_btn = gr.Button("🎲 Rastgele Tweet Getir", variant="secondary")
                etiketle_btn = gr.Button("▶ Etiketle", variant="primary")

            gr.Markdown("### 📊 Model Çıktısı")

            sonuc_box = gr.Markdown(
                value="*Etiketle butonuna basınca sonuç burada görünecek.*"
            )

    # ── Notlar ───────────────────────────────────────────────────────────────
    gr.Markdown(
        """---
        ### 💡 Prompt Engineering İpuçları
        - **Kategori tanımlarını netleştirin** — Model belirsiz tanımlarda hata yapar.
        - **Örnekler ekleyin** — `Örnek: 'Maaşım enflasyona yetişemiyor' → ekonomik_istikrarsizlik`
        - **Temperature'ı kademeli artırın** — 0.0'da başlayın, farkı gözlemleyin.
        - **Küçük model vs büyük model** — Maliyeti ve doğruluğu karşılaştırın.
        """
    )

    # ── Bağlantılar (event handlers) ─────────────────────────────────────────
    rastgele_btn.click(
        fn=rastgele_tweet_getir,
        inputs=[],
        outputs=[tweet_id_box, tweet_metin_box, gercek_etiket_box]
    )

    etiketle_btn.click(
        fn=etiketle_ve_goster,
        inputs=[tweet_metin_box, system_prompt_box, model_sec, temperature],
        outputs=[sonuc_box]
    )

    varsayilana_don.click(
        fn=lambda: VARSAYILAN_PROMPT,
        inputs=[],
        outputs=[system_prompt_box]
    )


# Arayüzü başlat
arayuz.launch(share=False)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


---

## 6. Toplu Etiketleme — Tüm Veri Setini İşleme

Prompt'unuzdan memnun olduktan sonra bu hücreyi çalıştırın.  
`SYSTEM_PROMPT` değişkenini yukarıdaki arayüzden geliştirdiğiniz prompt ile güncelleyin.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Arayüzde geliştirdiğiniz en iyi prompt'u buraya yapıştırın:
# ─────────────────────────────────────────────────────────────────────────────
KULLANILACAK_PROMPT =  """Sen bir sosyal bilim araştırma asistanısın. Görevin, Türkiye'deki genç
ve nitelikli iş gücüne ait tweetleri göç motivasyonu kategorilerine göre etiketlemektir.

ANA KATEGORİLER:
- "push" : Türkiye'yi terk etmeye iten nedenler (olumsuz koşullar)
- "pull" : Yurt dışını çekici kılan nedenler (olumlu fırsatlar)

ALT KATEGORİLER (push):
- ekonomik_istikrarsizlik  : Enflasyon, kur, geçim sıkıntısı
- liyakat_sorunu           : Torpil, adam kayırma, fırsat eşitsizliği
- ozgurluk_kisitlari       : Siyasi baskı, ifade özgürlüğü, akademik özgürlük
- hukuk_guvenlik_kaygisi   : Hukuk devleti, yargı bağımsızlığı
- egitim_sorunlari         : Eğitim kalitesi, akademik ortam
- girisimcilik_engelleri   : Bürokratik engeller, iş kurma güçlüğü
- zayif_sosyal_guvence     : Sağlık, emeklilik, sosyal haklar
- gelecek_belirsizligi     : Uzun vadeli belirsizlik, planlayamama
- dusuk_yasam_kalitesi     : Kentsel yaşam, çevre, altyapı

ALT KATEGORİLER (pull):
- daha_yuksek_gelir        : Maaş farkı, döviz kazancı
- kariyer_gelisimi         : Kariyer fırsatları, mentorluk, büyüme
- is_firsatlari            : Pozisyon çeşitliliği, istihdam
- yuksek_yasam_kalitesi    : Altyapı, çevre, şehir kalitesi
- refah_devleti            : Sosyal güvenlik, sağlık, haklar
- cocuklar_icin_gelecek    : Çocukların eğitimi, geleceği

Cevabını SADECE şu JSON formatında ver, başka hiçbir şey yazma:
{"ana_kategori": "push veya pull", "alt_kategori": "yukarıdan birini seç", "güven": "yüksek/orta/düşük", "gerekçe": "max 2 cümle"}"""


KULLANILACAK_MODEL       = "gpt-4o-mini"
KULLANILACAK_TEMPERATURE = 0.0

# Sadece ilk 20 tweet ile test edin — tam çalıştırmak için len(df) yapın
TEST_SATIR_SAYISI = 20

print(f"Etiketlenecek tweet sayısı : {TEST_SATIR_SAYISI}")
print(f"Model                      : {KULLANILACAK_MODEL}")
print(f"Temperature                : {KULLANILACAK_TEMPERATURE}")
print("-" * 50)

Etiketlenecek tweet sayısı : 20
Model                      : gpt-4o-mini
Temperature                : 0.0
--------------------------------------------------


In [17]:
sonuclar = []

for i, satir in df.head(TEST_SATIR_SAYISI).iterrows():
    print(f"[{i+1}/{TEST_SATIR_SAYISI}] Tweet #{satir['id']} işleniyor...", end=" ")

    etiket = tweet_etiketle(
        metin=satir["metin"],
        system_prompt=KULLANILACAK_PROMPT,
        model=KULLANILACAK_MODEL,
        temperature=KULLANILACAK_TEMPERATURE
    )

    sonuclar.append({
        "id"                  : satir["id"],
        "metin"               : satir["metin"],
        "gercek_ana"          : satir["ana_kategori"],
        "gercek_alt"          : satir["alt_kategori"],
        "model_ana"           : etiket.get("ana_kategori"),
        "model_alt"           : etiket.get("alt_kategori"),
        "güven"               : etiket.get("güven"),
        "gerekçe"             : etiket.get("gerekçe"),
        "token_kullanim"      : etiket.get("token_kullanim", 0),
    })

    # Ana kategori uyuşuyor mu?
    uyusma = "✓" if etiket.get("ana_kategori") == satir["ana_kategori"] else "✗"
    print(f"Ana: {uyusma} | Alt: {etiket.get('alt_kategori', 'HATA')}")

    time.sleep(0.3)  # Rate limit koruması

print("\n✓ Tüm tweetler etiketlendi!")

[1/20] Tweet #1 işleniyor... Ana: ✓ | Alt: ekonomik_istikrarsizlik
[2/20] Tweet #2 işleniyor... Ana: ✓ | Alt: ekonomik_istikrarsizlik
[3/20] Tweet #3 işleniyor... Ana: ✓ | Alt: ekonomik_istikrarsizlik
[4/20] Tweet #4 işleniyor... Ana: ✓ | Alt: ekonomik_istikrarsizlik
[5/20] Tweet #5 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[6/20] Tweet #6 işleniyor... Ana: ✓ | Alt: ekonomik_istikrarsizlik
[7/20] Tweet #7 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[8/20] Tweet #8 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[9/20] Tweet #9 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[10/20] Tweet #10 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[11/20] Tweet #11 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[12/20] Tweet #12 işleniyor... Ana: ✓ | Alt: gelecek_belirsizligi
[13/20] Tweet #13 işleniyor... Ana: ✓ | Alt: liyakat_sorunu
[14/20] Tweet #14 işleniyor... Ana: ✓ | Alt: liyakat_sorunu
[15/20] Tweet #15 işleniyor... Ana: ✓ | Alt: liyakat_sorunu
[16/20] Tweet #16 işleniyor... 

## 7. Sonuçları Analiz Et

Modelin performansını değerlendirelim:
- Ana kategori doğruluğu (`push` / `pull`)
- Alt kategori doğruluğu
- Hangi kategorilerde hata yüksek?

In [19]:
df_sonuc = pd.DataFrame(sonuclar)

# Doğruluk hesapla
df_sonuc["ana_dogru"] = df_sonuc["gercek_ana"] == df_sonuc["model_ana"]
df_sonuc["alt_dogru"] = df_sonuc["gercek_alt"] == df_sonuc["model_alt"]

ana_dogruluk = df_sonuc["ana_dogru"].mean() * 100
alt_dogruluk = df_sonuc["alt_dogru"].mean() * 100
toplam_token = df_sonuc["token_kullanim"].sum()

print("=" * 45)
print("PERFORMANS ÖZETİ")
print("=" * 45)
print(f"Ana kategori doğruluğu : %{ana_dogruluk:.1f}")
print(f"Alt kategori doğruluğu : %{alt_dogruluk:.1f}")
print(f"Toplam token kullanımı : {toplam_token}")
print(f"Tweet başına ort. token: {toplam_token / len(df_sonuc):.0f}")

PERFORMANS ÖZETİ
Ana kategori doğruluğu : %100.0
Alt kategori doğruluğu : %95.0
Toplam token kullanımı : 12523
Tweet başına ort. token: 626


In [21]:
# Kategori bazında doğruluk
print("Alt kategoriye göre doğruluk oranı:")
print("-" * 45)

kat_dogruluk = (
    df_sonuc.groupby("gercek_alt")["alt_dogru"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "doğru", "count": "toplam"})
)
kat_dogruluk["oran"] = (kat_dogruluk["doğru"] / kat_dogruluk["toplam"] * 100).round(1)
kat_dogruluk = kat_dogruluk.sort_values("oran")

for idx, row in kat_dogruluk.iterrows():
    bar    = "█" * int(row["oran"] / 10)
    durum  = "✓" if row["oran"] == 100 else ("△" if row["oran"] >= 50 else "✗")
    print(f"{durum} {idx:<30} %{row['oran']:.0f}  ({int(row['doğru'])}/{int(row['toplam'])}) {bar}")

Alt kategoriye göre doğruluk oranı:
---------------------------------------------
△ ekonomik_istikrarsizlik        %83  (5/6) ████████
✓ gelecek_belirsizligi           %100  (6/6) ██████████
✓ liyakat_sorunu                 %100  (6/6) ██████████
✓ ozgurluk_kisitlari             %100  (2/2) ██████████


In [23]:
# Hataları incele — prompt'u geliştirmek için
print("HATALI ETİKETLEMELER (Ana Kategori)")
print("=" * 60)

hatalar = df_sonuc[~df_sonuc["ana_dogru"]]

if len(hatalar) == 0:
    print("✓ Ana kategoride hiç hata yok!")
else:
    for _, satir in hatalar.iterrows():
        print(f"Tweet: {satir['metin'][:70]}...")
        print(f"  Gerçek  : {satir['gercek_ana']} / {satir['gercek_alt']}")
        print(f"  Model   : {satir['model_ana']} / {satir['model_alt']}")
        print(f"  Gerekçe : {satir['gerekçe']}")
        print()

HATALI ETİKETLEMELER (Ana Kategori)
✓ Ana kategoride hiç hata yok!


In [25]:
# Sonuçları CSV'ye kaydet
df_sonuc.to_csv("annotation_sonuclari.csv", index=False, encoding="utf-8-sig")
print("✓ Sonuçlar 'annotation_sonuclari.csv' olarak kaydedildi.")

# Son hali göster
df_sonuc[["id", "gercek_ana", "model_ana", "ana_dogru", "gercek_alt", "model_alt", "alt_dogru", "güven"]]

✓ Sonuçlar 'annotation_sonuclari.csv' olarak kaydedildi.


,id,gercek_ana,model_ana,ana_dogru,gercek_alt,model_alt,alt_dogru,güven
0,1,push,push,True,ekonomik_istikrarsizlik,ekonomik_istikrarsizlik,True,yüksek
1,2,push,push,True,ekonomik_istikrarsizlik,ekonomik_istikrarsizlik,True,yüksek
2,3,push,push,True,ekonomik_istikrarsizlik,ekonomik_istikrarsizlik,True,yüksek
3,4,push,push,True,ekonomik_istikrarsizlik,ekonomik_istikrarsizlik,True,yüksek
4,5,push,push,True,ekonomik_istikrarsizlik,gelecek_belirsizligi,False,yüksek
5,6,push,push,True,ekonomik_istikrarsizlik,ekonomik_istikrarsizlik,True,yüksek
6,7,push,push,True,gelecek_belirsizligi,gelecek_belirsizligi,True,yüksek
7,8,push,push,True,gelecek_belirsizligi,gelecek_belirsizligi,True,yüksek
8,9,push,push,True,gelecek_belirsizligi,gelecek_belirsizligi,True,orta
9,10,push,push,True,gelecek_belirsizligi,gelecek_belirsizligi,True,yüksek


---

## 🔑 Bu Dersin Ana Mesajları

| Kavram | Ne öğrendik? |
|--------|-------------|
| **API** | Chat arayüzü değil, servis — otomasyon ve tekrar edilebilirlik sağlar |
| **API Key Güvenliği** | Asla koda yazılmaz; `getpass` veya `.env` kullanılır |
| **System Prompt** | Modelin rolünü ve kurallarını tanımlar — tutarlılığın temelidir |
| **Temperature = 0** | Annotation görevleri için düşük temperature tercih edilir |
| **JSON Çıktı** | Yapılandırılmış çıktı → doğrudan analitik pipeline'a entegrasyon |
| **Hata Yönetimi** | Model her zaman mükemmel çıktı üretmez; kontrol mekanizması şarttır |
| **Doğruluk Analizi** | Üretilen etiketler bilimsel iddia taşır — ölçülmeli ve savunulmalıdır |

> **Son not:** LLM kullanmak sadece teknik bir işlem değil, metodolojik bir karardır.  
> Üretilen her etiket bir *araştırma iddiasıdır* — ve her iddia savunulabilir olmalıdır.